# Phase 3.4a — Build the Concept → Paragraph Bridge

**Buddhist RAG System — Clear Light of Bliss**

## The problem

After Phase 3.3, the Neo4j graph contains two **disconnected sub-graphs**:

```
[Book] → [Chapter] → [Paragraph]         ← structural tree (3,533 nodes)

[Concept] ⟷ [Concept] ⟷ [Concept]        ← doctrinal web (50 concepts, 683 rels)
```

There are no edges connecting Concept nodes to Paragraph nodes. That makes
hybrid retrieval (the goal of Phase 4) impossible — we can't expand from a
matched concept to its source paragraphs.

## What this notebook does

Adds `MENTIONS` edges from each Concept to the Paragraphs that mention it,
using evidence from `07_semantic_relationships.json` (the Phase 3 NLP output
which already records the source paragraph for every extracted relationship).

## Process — validation-gated

1. **Load** the source JSON and extract (concept, paragraph_citation) pairs.
2. **Validate** that the citation format in the JSON matches the format in Neo4j.
3. **Dry-run** preview the edges that would be created.
4. **Write** the `MENTIONS` edges using `MERGE` (idempotent).
5. **Verify** with the same query that previously returned zero results.

Each step prints its result. We don't write until validation passes.

---
## Setup — imports and Neo4j connection

We're using two libraries: standard `json` for the file, and the official
`neo4j` Python driver for the graph. The driver talks Bolt protocol on port
7687 (different from the Browser's HTTP port 7474, but same database).

**Read-only at this stage.** We just open the connection and confirm it works.

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

SEMANTIC_RELS_PATH = Path('07_semantic_relationships.json')

# Confirm connection by counting nodes (read-only)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    result = session.run('MATCH (n) RETURN count(n) AS total').single()
    print(f'Connected to Neo4j. Total nodes: {result["total"]}')
    
    result = session.run('MATCH ()-[r]->() RETURN count(r) AS total').single()
    print(f'Total relationships: {result["total"]}')
    
    result = session.run(
        "MATCH ()-[r:MENTIONS]->() RETURN count(r) AS existing_mentions"
    ).single()
    print(f'Existing MENTIONS edges: {result["existing_mentions"]} '
          f'(should be 0 if this is the first run)')

---
## Step 1 — Load and extract (concept, paragraph) pairs

Every relationship record in `07_semantic_relationships.json` looks roughly like:

```json
{
  "subject": "clear_light",
  "relation": "depends_upon",
  "object": "emptiness",
  "source": {
    "paragraph_id": "clb_ch10_para103",
    "citation": "CLB.10.§2.p103",
    "chapter_index": 10
  }
}
```

Both `subject` and `object` are concept names, and `source.citation` is the
paragraph where this relationship was extracted from. So **each relationship
record gives us two (concept, paragraph) pairs**: one for the subject, one for
the object.

We collect all pairs and deduplicate.

In [ ]:
with open(SEMANTIC_RELS_PATH) as f:
    sem = json.load(f)

relationships = sem['relationships']
print(f'Loaded {len(relationships)} extracted relationships')

# Build a map: concept → set of paragraph citations
concept_to_paragraphs = defaultdict(set)
missing_citation_count = 0

for rel in relationships:
    citation = rel.get('source', {}).get('citation')
    if not citation:
        missing_citation_count += 1
        continue
    for role in ('subject', 'object'):
        concept = rel.get(role)
        if concept:
            concept_to_paragraphs[concept].add(citation)

# Flatten into a list of (concept, paragraph) pairs
pairs = [(c, p) for c, paras in concept_to_paragraphs.items() for p in paras]

print(f'Unique concepts seen:  {len(concept_to_paragraphs)}')
print(f'Unique (concept, paragraph) pairs: {len(pairs)}')
print(f'Relationships with missing citation: {missing_citation_count}')
print()
print('Sample — clear_light is mentioned in these paragraphs (first 5):')
for cit in sorted(list(concept_to_paragraphs['clear_light']))[:5]:
    print(f'  {cit}')
print()
print('Top 5 concepts by paragraph spread:')
top = sorted(concept_to_paragraphs.items(), key=lambda kv: -len(kv[1]))[:5]
for concept, paras in top:
    print(f'  {concept:20s} appears in {len(paras):4d} paragraphs')

---
## Step 2 — VALIDATION GATE: do these citations match the graph?

The JSON has citations like `CLB.10.§2.p103` — with a section marker `§2`.
Earlier in the Neo4j Browser we saw citations like `CLB.6.p2` — no section
marker. This could mean any of:

- (a) Both formats are used — chapters with sections get `§N`, chapters without get nothing. ✅ No problem; sample matches both.
- (b) The graph strips section markers — JSON `CLB.10.§2.p103` would not match graph's `CLB.10.p103`. ❌ Need to rewrite citations before writing edges.
- (c) Different paragraph numbering between JSON and graph. ❌ Need a deeper fix.

We sample 20 paragraph citations from the JSON and check whether Neo4j has them.

**If validation fails, we stop here.** Better to find out now than after writing
thousands of broken edges.

In [ ]:
import random

all_citations = list({p for _, p in pairs})
sample_size = min(20, len(all_citations))
random.seed(42)
sample = random.sample(all_citations, sample_size)

print(f'Sampling {sample_size} citations from the JSON and looking them up in Neo4j:')
print()

found = 0
missing = []
with driver.session() as session:
    for cit in sample:
        result = session.run(
            'MATCH (p:Paragraph {citation: $cit}) RETURN p.citation AS c',
            cit=cit
        ).single()
        if result:
            found += 1
            marker = '✓'
        else:
            missing.append(cit)
            marker = '✗'
        print(f'  {marker} {cit}')

print()
print(f'Found in graph: {found}/{sample_size}')

VALIDATION_PASSED = (found == sample_size)
if VALIDATION_PASSED:
    print('✓ Validation passed — citation formats match. Safe to proceed.')
else:
    print(f'✗ Validation failed — {len(missing)} citations not found in graph.')
    print('  Missing examples:')
    for m in missing[:5]:
        print(f'    {m}')
    print('  DO NOT proceed to the write step. Investigate the format mismatch first.')

---
## Step 3 — Dry run: preview what we'd write

Before any mutation, print the first 10 edges we'd create and a summary. This
lets us look at the data once more in its final shape and catch surprises.

Also: do we have any concepts in the JSON that **aren't in the graph as Concept
nodes**? If so, those pairs would silently fail (the `MATCH` half of the `MERGE`
wouldn't find the Concept and no edge would be created).

In [ ]:
# Get the set of canonical_forms that actually exist as Concept nodes
with driver.session() as session:
    result = session.run('MATCH (c:Concept) RETURN c.canonical_form AS cf')
    graph_concepts = {r['cf'] for r in result}

json_concepts = set(concept_to_paragraphs.keys())

in_both = json_concepts & graph_concepts
only_json = json_concepts - graph_concepts
only_graph = graph_concepts - json_concepts

print(f'Concepts in JSON: {len(json_concepts)}')
print(f'Concepts in graph: {len(graph_concepts)}')
print(f'Concepts in both (edges will be created): {len(in_both)}')
print(f'Concepts in JSON only (edges will be skipped): {len(only_json)}')
print(f'Concepts in graph only (no MENTIONS edges to add): {len(only_graph)}')

if only_json:
    print(f'  JSON-only examples: {sorted(only_json)[:10]}')
if only_graph:
    print(f'  Graph-only examples: {sorted(only_graph)[:10]}')

# Filter pairs to only those where the concept exists in the graph
valid_pairs = [(c, p) for c, p in pairs if c in graph_concepts]
print()
print(f'Pairs to be written: {len(valid_pairs)} (of {len(pairs)} total)')
print()
print('First 10 pairs that would be written:')
for c, p in valid_pairs[:10]:
    print(f'  ({c}) -[:MENTIONS]-> ({p})')

---
## Step 4 — Write the `MENTIONS` edges

Now we actually mutate the graph. A few choices to explain:

**Why `MERGE` instead of `CREATE`?** `MERGE` is "create if not exists" — it
looks for an existing edge with the same endpoints and only creates one if
absent. This makes the notebook **idempotent**: running it twice produces the
same final state as running it once. `CREATE` would make duplicate edges on a
second run.

**Why batched and parameterized?** We use `UNWIND $pairs AS pair` to send all
the pairs in one query as a parameter, instead of running thousands of separate
queries. This is the standard Neo4j idiom for bulk writes. Much faster.

**Why batched at 200?** Single very large parameter lists can exceed transaction
memory. 200 at a time is comfortable.

**Why explicit `WITH driver.session() as ... session.execute_write()`?**
`execute_write` wraps each call in a transaction. If anything inside fails, the
transaction rolls back and the graph is left untouched.

In [ ]:
if not VALIDATION_PASSED:
    raise RuntimeError('Validation failed in Step 2 — refusing to write.')

BATCH_SIZE = 200

WRITE_QUERY = '''
UNWIND $pairs AS pair
MATCH (c:Concept {canonical_form: pair.concept})
MATCH (p:Paragraph {citation: pair.citation})
MERGE (c)-[r:MENTIONS]->(p)
RETURN count(r) AS edges_in_batch
'''

# Convert to the dict-of-records shape Neo4j expects for UNWIND
pair_records = [{'concept': c, 'citation': p} for c, p in valid_pairs]

total_written = 0
with driver.session() as session:
    for i in range(0, len(pair_records), BATCH_SIZE):
        batch = pair_records[i:i + BATCH_SIZE]
        result = session.execute_write(
            lambda tx: tx.run(WRITE_QUERY, pairs=batch).single()
        )
        edges = result['edges_in_batch']
        total_written += edges
        print(f'  Batch {i // BATCH_SIZE + 1}: {edges} edges (running total: {total_written})')

print()
print(f'Done. Reported total: {total_written}')

# Independent count for cross-check
with driver.session() as session:
    actual = session.run(
        'MATCH ()-[r:MENTIONS]->() RETURN count(r) AS c'
    ).single()['c']
print(f'MENTIONS edges in graph now: {actual}')
print(f'Expected (deduplicated pairs): {len(valid_pairs)}')

---
## Step 5 — Verify visually (the "Show me" moment)

We re-run the query that returned zero results earlier — this time, it should
light up.

Open the Neo4j Browser at <http://localhost:7474> and run:

```cypher
MATCH (c:Concept {canonical_form: 'clear_light'})-[r:MENTIONS]->(p:Paragraph)
RETURN c, r, p LIMIT 25
```

You should see `clear_light` in the center with up to 25 Paragraph nodes
radiating out, each connected by a `MENTIONS` edge.

Below we run the same query from Python and print a summary.

In [ ]:
verify_query = '''
MATCH (c:Concept {canonical_form: $concept})-[:MENTIONS]->(p:Paragraph)
RETURN p.citation AS citation, p.chapter_index AS chapter, 
       substring(p.text, 0, 100) AS preview
ORDER BY p.chapter_index, p.paragraph_index
LIMIT 10
'''

for concept in ['clear_light', 'emptiness', 'mahamudra']:
    print(f'=== {concept} ===')
    with driver.session() as session:
        result = session.run(verify_query, concept=concept)
        rows = list(result)
    if not rows:
        print('  (no mentions)')
    for row in rows:
        print(f'  {row["citation"]:25s} ch{row["chapter"]:2d}  {row["preview"]}')
    print()

# Cross-check: per-concept paragraph counts should match what we computed in Step 1
print('=== Sanity check: per-concept paragraph counts (graph vs JSON) ===')
with driver.session() as session:
    result = session.run('''
        MATCH (c:Concept)-[:MENTIONS]->(p:Paragraph)
        RETURN c.canonical_form AS concept, count(p) AS graph_count
        ORDER BY graph_count DESC LIMIT 10
    ''')
    for row in result:
        c = row['concept']
        graph_count = row['graph_count']
        json_count = len(concept_to_paragraphs.get(c, set()))
        match = '✓' if graph_count == json_count else '✗'
        print(f'  {match} {c:20s}  graph={graph_count:4d}  json={json_count:4d}')

---
## Done

The Concept layer and the structural layer are now connected. The graph went
from two disconnected sub-graphs to one unified graph that supports hybrid
retrieval.

**What's now possible that wasn't before:**

- Ask "what paragraphs mention `clear_light`?" → traverse one edge.
- Ask "what other concepts appear in the same paragraphs as `clear_light`?" → 
  two-hop traversal: `(:Concept)-[:MENTIONS]->(:Paragraph)<-[:MENTIONS]-(:Concept)`.
- Ask "give me all paragraphs that mention any concept related to `emptiness`
  via `DEPENDS_UPON`" → three-hop traversal.

These multi-hop queries are what graph databases exist for — and they're now
queries we can write against your corpus.

**Next phase:** 4.1 — design the hybrid retrieval interface that uses both this
graph and ChromaDB.

In [ ]:
driver.close()
print('Neo4j connection closed.')